In [1]:
!pip install transformers torch scikit-learn

In [4]:
# ============================================
# TRANSFORMER NLP PIPELINE (DETAILED)
# ============================================

import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# ============================================
# 1. REPRODUCIBILITY (IMPORTANT)
# ============================================
torch.manual_seed(42)
np.random.seed(42)

# ============================================
# 2. LOAD MODEL + TOKENIZER
# ============================================
model_name = "distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

model.eval()  # disable dropout

# ============================================
# 3. INPUT DATASET
# ============================================
texts = [
    "I love this product, it is amazing!",
    "This is the worst experience ever.",
    "I hate this service."
]

# Ground truth (manually defined)
true_labels = ["POSITIVE", "NEGATIVE", "NEGATIVE"]

print("========================================")
print("INPUT TEXTS")
print("========================================")
for i, t in enumerate(texts):
    print(f"{i+1}. {t}")

# ============================================
# 4. TOKENIZATION (DETAILED)
# ============================================
print("\n========================================")
print("TOKENIZATION DETAILS (FIRST SENTENCE)")
print("========================================")

encoded = tokenizer(
    texts[0],
    padding=True,
    truncation=True,
    return_tensors="pt"
)

print("Input IDs:", encoded["input_ids"])
print("Attention Mask:", encoded["attention_mask"])

# Convert IDs back to tokens
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])
print("Tokens:", tokens)

# ============================================
# 5. MODEL INFERENCE (LOW-LEVEL)
# ============================================
print("\n========================================")
print("MODEL INFERENCE (MANUAL COMPUTATION)")
print("========================================")

pred_labels = []

for text in texts:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    probs = torch.softmax(logits, dim=1)
    pred_class = torch.argmax(probs, dim=1).item()

    label = "POSITIVE" if pred_class == 1 else "NEGATIVE"
    confidence = probs[0][pred_class].item()

    pred_labels.append(label)

    print(f"\nText: {text}")
    print("Logits:", logits.numpy())
    print("Probabilities:", probs.numpy())
    print("Prediction:", label)
    print("Confidence:", round(confidence, 4))

# ============================================
# 6. PIPELINE (HIGH-LEVEL COMPARISON)
# ============================================
print("\n========================================")
print("PIPELINE OUTPUT (COMPARISON)")
print("========================================")

classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

pipe_results = classifier(texts)

for t, r in zip(texts, pipe_results):
    print(f"\nText: {t}")
    print(f"Pipeline Prediction: {r['label']} ({round(r['score'],4)})")

# ============================================
# 7. EVALUATION
# ============================================
print("\n========================================")
print("EVALUATION METRICS")
print("========================================")

# Convert labels to numeric
label_map = {"NEGATIVE": 0, "POSITIVE": 1}

y_true = [label_map[l] for l in true_labels]
y_pred = [label_map[l] for l in pred_labels]

# Accuracy
acc = accuracy_score(y_true, y_pred)
print("Accuracy:", acc)

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
print("\nConfusion Matrix:\n", cm)

# Classification Report
report = classification_report(y_true, y_pred, target_names=["NEGATIVE", "POSITIVE"])
print("\nClassification Report:\n", report)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

INPUT TEXTS
1. I love this product, it is amazing!
2. This is the worst experience ever.
3. I hate this service.

TOKENIZATION DETAILS (FIRST SENTENCE)
Input IDs: tensor([[ 101, 1045, 2293, 2023, 4031, 1010, 2009, 2003, 6429,  999,  102]])
Attention Mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
Tokens: ['[CLS]', 'i', 'love', 'this', 'product', ',', 'it', 'is', 'amazing', '!', '[SEP]']

MODEL INFERENCE (MANUAL COMPUTATION)

Text: I love this product, it is amazing!
Logits: [[-4.3702626  4.715618 ]]
Probabilities: [[1.1324086e-04 9.9988675e-01]]
Prediction: POSITIVE
Confidence: 0.9999

Text: This is the worst experience ever.
Logits: [[ 4.6606965 -3.7283964]]
Probabilities: [[9.997727e-01 2.272816e-04]]
Prediction: NEGATIVE
Confidence: 0.9998

Text: I hate this service.
Logits: [[ 4.432876  -3.5761645]]
Probabilities: [[9.9966764e-01 3.3233294e-04]]
Prediction: NEGATIVE
Confidence: 0.9997

PIPELINE OUTPUT (COMPARISON)

Text: I love this product, it is amazing!
Pipeline Prediction: PO